# Project: AI Reply Assistant Agent

What this project will do

You’ll paste an incoming email/message into the notebook, for example:

recruiter message
interview invitation
follow-up email
leave-related message
professional chat message

The project will then:

Step 1 — clean the message
using message_parser.py

Step 2 — detect the likely intent
using intent_classifier.py

Step 3 — build a structured prompt
using prompt_builder.py

Step 4 — send it to Ollama
from the notebook

Step 5 — generate a polished reply
and display it in markdown

Cell 1 — Import project modules and notebook display tools

In [13]:
from ollama import Client
from IPython.display import Markdown, display
from openai import OpenAI
import message_parser
import intent_classifier
import prompt_builder

Cell 2 — Connect to local Ollama and set the model

In [22]:

OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama =  OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
model = "llama3.2:1b"

print(f"Reply Assistant is ready. Using model: {model}")

Reply Assistant is ready. Using model: llama3.2:1b


Cell 3 — Add a sample incoming message

In [23]:
raw_message = """
Hi Faizan,

We came across your profile and would like to schedule a quick discussion for a Frontend Developer opportunity.
Your background seems relevant to one of our current openings.

Please share your availability for this week if you are interested.

Best regards,
Recruitment Team
"""

Cell 4 — Parse and clean the incoming message

In [24]:
parsed_message = message_parser.prepare_message_input(raw_message)

cleaned_message = parsed_message["cleaned_message"]
message_stats = parsed_message["stats"]

print("Cleaned Message:\n")
print(cleaned_message)

print("\nMessage Stats:")
print(message_stats)

Cleaned Message:

Hi Faizan,

We came across your profile and would like to schedule a quick discussion for a Frontend Developer opportunity.
Your background seems relevant to one of our current openings.

Please share your availability for this week if you are interested.

Best regards,
Recruitment Team

Message Stats:
{'word_count': 45, 'line_count': 6, 'char_count': 287}


Cell 5 — Detect intent and reply goal
What this cell does

This cell uses intent_classifier.py to:

detect what kind of message this is
decide the likely reply goal

In [25]:
detected_intent = intent_classifier.detect_intent(cleaned_message)
reply_goal = intent_classifier.suggest_reply_goal(detected_intent)

print("Detected Intent:", detected_intent)
print("Reply Goal:", reply_goal)

Detected Intent: interview_invitation
Reply Goal: accept_and_share_availability


Cell 6 — Build the prompts
What this cell does

This cell builds:

the reusable system prompt
the user prompt for this specific incoming message

In [26]:
system_prompt = prompt_builder.build_system_prompt()

user_prompt = prompt_builder.build_user_prompt(
    cleaned_message=cleaned_message,
    detected_intent=detected_intent,
    reply_goal=reply_goal,
    tone="professional"
)

print("User Prompt Preview:\n")
print(user_prompt)

User Prompt Preview:

Please write a reply to the following incoming message.

Detected Intent: interview_invitation
Suggested Reply Goal: accept_and_share_availability
Preferred Tone: professional

Incoming Message:
Hi Faizan,

We came across your profile and would like to schedule a quick discussion for a Frontend Developer opportunity.
Your background seems relevant to one of our current openings.

Please share your availability for this week if you are interested.

Best regards,
Recruitment Team


Cell 7 — Create the chat messages

In [27]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

Cell 8 — Send the request to the model

In [28]:
response = ollama.chat.completions.create(model=model, messages=messages)

Cell 9 — Extract the generated reply

In [30]:
output = response.choices[0].message.content
print(output)

# Reply Draft
Hi Faizan, 

Thank you for considering me for the frontend developer position at [Company Name]. We're excited about the opportunity and believe my skills align with one of our current openings. I'm glad to hear that my background is relevant.

I'd prefer to schedule a call to discuss further if possible. Could you please let me know when this would work best for you?

Best regards,
[Your Name]


In [31]:
display(Markdown(output))

# Reply Draft
Hi Faizan, 

Thank you for considering me for the frontend developer position at [Company Name]. We're excited about the opportunity and believe my skills align with one of our current openings. I'm glad to hear that my background is relevant.

I'd prefer to schedule a call to discuss further if possible. Could you please let me know when this would work best for you?

Best regards,
[Your Name]